# 02 · Training and calibration

A model that keeps its accuracy under a perturbation but becomes overconfident has still degraded.
Accuracy alone will not show that, so calibration is measured from the start.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

sys.path.insert(0, str(Path.cwd().parent / "src"))  # works without installing the package

CONFIG = "../configs/default.yaml"   # switch to ../configs/smoke.yaml to run offline in seconds


In [ ]:
from ctxaiqc.data import load_dataset
from ctxaiqc.train import train_model
from ctxaiqc.utils import load_config

cfg = load_config(CONFIG)
data = load_dataset(**cfg["dataset"], seed=cfg["seed"])

# Skip if a checkpoint already exists; training is also `make train` from the shell.
if not Path(cfg["train"]["checkpoint"]).exists():
    history = train_model(cfg)

In [ ]:
from ctxaiqc.evaluate import load_checkpoint, predict_probs
from ctxaiqc.metrics import expected_calibration_error, summarise_performance

model, ckpt = load_checkpoint(cfg["train"]["checkpoint"])
probs = predict_probs(model, data.x_test)

scores = summarise_performance(data.y_test, probs)
scores["ece"] = expected_calibration_error(data.y_test, probs)
scores

## Reliability diagram\n\nPerfect calibration is the diagonal. Above it the model is underconfident, below it overconfident.

In [ ]:
from ctxaiqc.metrics import reliability_curve

centres, acc, conf, weights = reliability_curve(data.y_test, probs, n_bins=12)

fig, ax = plt.subplots(figsize=(4.6, 4.2))
ax.plot([0, 1], [0, 1], "--", color="grey", lw=1, label="perfect calibration")
ax.plot(conf, acc, "o-", label="model")
ax.set_xlabel("confidence"); ax.set_ylabel("accuracy")
ax.set_title(f"ECE = {scores['ece']:.3f}")
ax.legend(); ax.grid(alpha=0.3)

## Monte-Carlo dropout

Dropout is left active at test time and the forward pass is repeated. The spread of the softmax
across passes is a cheap, distribution-level view of predictive uncertainty that costs no retraining.

In [ ]:
from ctxaiqc.metrics import mc_dropout_probs

mean, std = mc_dropout_probs(model, data.x_test[:400], n_samples=20)
predicted_std = std[np.arange(len(mean)), mean.argmax(axis=1)]
correct = mean.argmax(axis=1) == data.y_test[:400]

fig, ax = plt.subplots(figsize=(5, 3.4))
ax.hist(predicted_std[correct], bins=25, alpha=0.7, label="correct", density=True)
ax.hist(predicted_std[~correct], bins=25, alpha=0.7, label="incorrect", density=True)
ax.set_xlabel("MC-dropout std of the predicted class"); ax.set_ylabel("density")
ax.legend(); ax.grid(alpha=0.3)

If the two histograms separate, the uncertainty carries usable information about when the model is
wrong. If they overlap completely, it does not — and that is worth knowing before anyone proposes
using it as a flag for review.